# Phân tích giỏ hàng bằng Eclat

Thí nghiệm dùng [UCI Online Retail](https://doi.org/10.24432/C5BW33) để tìm những mặt hàng thường xuất hiện trong cùng hóa đơn. Phần khai phá itemset và sinh luật sử dụng mã Julia trong repository.

## Môi trường và tham số

Chạy các ô theo thứ tự. Có thể đặt biến môi trường `JULIA_EXE` để dùng Julia từ đường dẫn riêng.

## Thư viện sử dụng

Thuật toán Eclat do nhóm tự cài đặt (`EclatMining`), không dùng thư viện khai thác tập phổ biến có sẵn. Các thư viện còn lại chỉ hỗ trợ cài môi trường, tải, đọc và kiểm tra dữ liệu, tính ngưỡng thống kê, hiển thị kết quả và vẽ hình; không thư viện nào tham gia khai thác itemset.

| Thư viện | Vai trò | Phạm vi sử dụng |
|---|---|---|
| `EclatMining` (của nhóm) | Khai thác itemset phổ biến bằng bản bitset | 1 và 2 |
| `XLSX.jl` (Julia) | Đọc workbook Excel của UCI Online Retail | 1 |
| `ZipFile.jl` (Julia) | Mở gói `.zip` tải từ UCI để lấy tệp `Online Retail.xlsx` | 1 |
| `Downloads`, `SHA` (thư viện chuẩn Julia) | Tải dữ liệu gốc khi còn thiếu và kiểm tra SHA-256 | Chuẩn bị benchmark và cả hai ứng dụng |
| `TOML` (thư viện chuẩn Julia) | Đọc cấu hình benchmark từ `data/datasets.toml` và ghi thông tin lần chạy splice vào `run_info.toml` | Chuẩn bị benchmark và ứng dụng 2 |
| `subprocess`, `tarfile`, `urllib.request`, `shutil`, `tempfile`, `os`, `pathlib`, `platform` (Python) | Cài Julia trong notebook và chạy các script Julia | 1 và 2, cùng bước chuẩn bị benchmark |
| `csv`, `json`, `hashlib` (Python) | Đọc các bảng kết quả đã lưu, đối chiếu với bản tái tạo, kiểm tra SHA-256 | 2 |
| `statistics.NormalDist` (Python) | Tính ngưỡng χ² sau hiệu chỉnh Bonferroni | 2 |
| `IPython.display` (Python) | Hiển thị bảng và hình trong notebook | 1 và 2 |
| `matplotlib` (Python, trong `scripts/plot_splice_patterns.py`) | Vẽ hai hình của ứng dụng 2 để notebook hiển thị từ các tệp đã lưu | 2 |

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import tarfile
import tempfile
from urllib.request import urlretrieve

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts/run_market_basket.jl").is_file()),
    None,
)
if repo is None:
    repo = Path.cwd() / "eclat-mining"
    if not repo.exists():
        subprocess.run([
            "git", "clone", "https://github.com/AIVIETNAM-AIO-Heart208/eclat-mining.git", str(repo)
        ], check=True)
    if not (repo / "scripts/run_market_basket.jl").is_file():
        raise FileNotFoundError("Cần phiên bản repository có scripts/run_market_basket.jl.")

julia = os.environ.get("JULIA_EXE") or shutil.which("julia")
if julia is None:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Cài Julia 1.9 trở lên hoặc khai báo JULIA_EXE.")
    runtime = Path(tempfile.gettempdir()) / "eclat-julia-runtime"
    runtime.mkdir(exist_ok=True)
    executable = runtime / "julia-1.11.4/bin/julia"
    if not executable.is_file():
        with tempfile.TemporaryDirectory() as download:
            archive = Path(download) / "julia.tar.gz"
            urlretrieve("https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz", archive)
            with tarfile.open(archive) as source:
                source.extractall(runtime, filter="data")
    julia = str(executable)

subprocess.run([julia, "--version"], check=True)
subprocess.run([julia, f"--project={repo}", "-e", "using Pkg; Pkg.instantiate()"], cwd=repo, check=True)
minsup = "1%"
minconf = "50%"

## Chuẩn bị dữ liệu benchmark

Script Julia tải các file FIMI còn thiếu, kiểm tra SHA-256 và chuẩn hóa năm bộ Chess, Mushroom, Retail, Accidents và T10I4D100K. Các file giao dịch đã chuẩn hóa nằm trong `data/processed/` của repository.

In [ ]:
benchmark_run = subprocess.run([
    julia, f"--project={repo}", str(repo / "scripts/prepare_benchmarks.jl"),
], cwd=repo, capture_output=True, text=True, encoding="utf-8")
print(benchmark_run.stdout, benchmark_run.stderr, sep="", end="")
benchmark_run.check_returncode()
print(f"Dữ liệu benchmark đã chuẩn hóa tại {repo / 'data/processed'}")

## Từ hóa đơn đến luật kết hợp

Julia dùng XLSX.jl để đọc trực tiếp từng dòng trong workbook gốc sau khi xác minh SHA-256. Gom các dòng hợp lệ theo `InvoiceNo`, giữ mỗi `StockCode` một lần trong giỏ và ánh xạ mã sang ID nguyên theo thứ tự từ điển.

Loại hóa đơn hủy, dòng thiếu mã, số lượng hoặc đơn giá không hợp lệ và các mã phí hay điều chỉnh đã xác định. Kết quả dưới đây gồm số dòng, hóa đơn, mã hàng qua từng bước và ví dụ hóa đơn 536365.

Eclat tìm itemset theo minsup. Chương trình sinh luật có hướng, lọc theo minconf, xếp hạng theo lift và đếm trực tiếp trên các giỏ để kiểm tra lại 10 luật được hiển thị.

In [ ]:
from IPython.display import Markdown, display

with tempfile.TemporaryDirectory() as directory:
    output = Path(directory) / "online_retail_rules.md"
    subprocess.run([
        julia, "--startup-file=no", "--threads=1", f"--project={repo}",
        str(repo / "scripts/run_market_basket.jl"), str(output), minsup, minconf,
    ], cwd=repo, check=True)
    display(Markdown(output.read_text(encoding="utf-8")))

## Diễn giải

Với minsup 1% và minconf 50%, các luật có lift cao nhất tập trung vào thẻ đánh dấu cây gia vị. Hai chiều của một luật có cùng lift nhưng có thể khác confidence vì số giỏ ở mẫu số khác nhau. Có thể dùng kết quả để chọn bộ sản phẩm bán kèm rồi kiểm tra hiệu quả trên dữ liệu mới.

[Báo cáo phân tích giỏ hàng](../docs/report/market_basket.md) trình bày số liệu và diễn giải từng nhóm luật ở ngưỡng mặc định.

# Ứng dụng 2: mẫu DNA quanh vị trí cắt nối gen

Phần này dùng bộ [UCI Molecular Biology (Splice-junction Gene Sequences)](https://doi.org/10.24432/C5M888) (CC BY 4.0): 3190 đoạn DNA linh trưởng, mỗi đoạn 60 nucleotide quanh một vị trí. Nhãn EI là ranh giới exon → intron (donor), IE là ranh giới intron → exon (acceptor), N là đoạn không chứa vị trí cắt nối, dùng làm nhóm đối chứng.

Câu hỏi: quanh vị trí cắt nối, những nucleotide ở những vị trí nào thường xuất hiện cùng nhau, và chúng khác đoạn DNA bình thường ra sao. Quy tắc GT-AG (intron thường bắt đầu bằng GT, kết thúc bằng AG) và các mẫu chuẩn trong file `splice.theory` đi kèm bộ dữ liệu là đáp án để đối chiếu.

## Chạy lại toàn bộ phân tích

`scripts/splice_patterns.jl` tải ba file từ UCI vào `data/raw/`, kiểm SHA-256, bỏ 184 đoạn trùng trong cùng lớp và 2 đoạn mang nhãn mâu thuẫn, mã hóa theo hai cách, khai phá từng lớp bằng Eclat của nhóm rồi ghi các bảng kết quả. Ô dưới ghi vào một thư mục tạm, không ghi đè kết quả đã commit trong `results/applications/splice/`.

In [ ]:
import csv
import hashlib
from IPython.display import Image, Markdown, display

if not (repo / "scripts/splice_patterns.jl").is_file():
    raise FileNotFoundError("Cần phiên bản repository có scripts/splice_patterns.jl.")

splice_out = Path(tempfile.mkdtemp(prefix="splice_"))
run = subprocess.run([
    julia, "--startup-file=no", "--threads=1", f"--project={repo}",
    str(repo / "scripts/splice_patterns.jl"), "--output-dir", str(splice_out),
], cwd=repo, capture_output=True, text=True, encoding="utf-8")
print(run.stdout, run.stderr, sep="")
run.check_returncode()


def read_rows(name, folder=splice_out):
    with open(folder / name, newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def show_table(header, rows):
    escape = lambda value: str(value).replace("|", "\\|")
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(escape(value) for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def percent(value):
    return f"{100 * float(value):.1f}%"

## Từ chuỗi DNA đến giao dịch

Mỗi đoạn DNA là một giao dịch. Item là cặp (vị trí, nucleotide), ví dụ `+1G` là G ngay sau ranh giới và `-3C` là C ở vị trí thứ ba trước ranh giới; vị trí đánh số -30..-1, +1..+30, không có 0. Mỗi chuỗi đầu vào dài 60 nucleotide. Mã hóa A/C/G/T có 240 item khả dĩ. Mỗi giao dịch có tối đa 60 item vì các vị trí có ký tự mơ hồ được bỏ qua. Item phải gắn vị trí vì tín hiệu cắt nối nằm ở vị trí cố định so với ranh giới: trên dữ liệu gốc, 93% đoạn N có "GT" ở đâu đó; còn GT đúng tại +1, +2 (tính trên dữ liệu đã làm sạch) có ở 98,8% đoạn donor và chỉ 5,9% đoạn N.

Eclat nhận item là số nguyên: ký tự thứ j (1..60) mang nucleotide thứ s (A=1, C=2, G=3, T=4) thành item (j − 1) × 4 + s, nên giao dịch tự nhiên đã sắp tăng dần. Mã hóa thứ hai thay nucleotide bằng nhóm purin R (A, G) hoặc pyrimidin Y (C, T), có 120 item. Ký tự mơ hồ (N, D, R, S trong dữ liệu gốc) không sinh item ở mã hóa A/C/G/T; với R/Y, ký tự R và Y vẫn sinh item vì có nghĩa xác định.

Ô dưới mã hóa đoạn donor đầu tiên theo cả hai cách và in các item quanh ranh giới.

In [ ]:
encode_demo = '''
using EclatMining
records = read_splice(joinpath(ARGS[1], "data", "raw", "splice.data"); window=60)
r = first(filter(r -> r.class == "EI", records))
println("Đoạn ", r.name, ", nhãn ", r.class)
println("Vị trí -30..-1: ", r.seq[1:30])
println("Vị trí +1..+30: ", r.seq[31:60])
for enc in (NucleotideEncoding(60), PurinePyrimidineEncoding(60))
    t = encode(enc, r.seq)
    near = filter(i -> -3 <= parse(Int, item_label(enc, i)[1:end-1]) <= 6, t)
    println(nameof(typeof(enc)), ": giao dịch có ", length(t), " item; quanh ranh giới (-3..+6): ",
            join((string(item_label(enc, i), " = ", i) for i in near), ", "))
end
'''
with tempfile.TemporaryDirectory() as directory:
    script = Path(directory) / "encode_demo.jl"
    script.write_text(encode_demo, encoding="utf-8")
    demo = subprocess.run(
        [julia, "--startup-file=no", f"--project={repo}", str(script), str(repo)],
        cwd=repo, capture_output=True, text=True, encoding="utf-8",
    )
print(demo.stdout, demo.stderr, sep="")
demo.check_returncode()

## Chọn minsup

Khai phá riêng từng lớp với cùng minsup tương đối trong lớp, nên minsup có nghĩa trực tiếp, ví dụ "có ở ít nhất 30% đoạn donor". Lớp N được xử lý y hệt và đóng vai trò đối chứng.

Ở một vị trí bất kỳ, mỗi nucleotide chiếm khoảng 1/4 số đoạn nếu chọn ngẫu nhiên (với R/Y là 1/2). Một item chỉ phổ biến ở mức nền này không nói lên điều gì, nên minsup phải cao hơn mức nền. Bảng dưới cho thấy 30% (A/C/G/T) và 60% (R/Y) là mức đầu tiên trong dải đã chạy mà lớp N không còn item phổ biến nào. Hạ thấp hơn thì lớp N bắt đầu có item đơn đạt ngưỡng; nâng lên 40% thì mất `-3C` của mẫu chuẩn donor (có ở 39,0% đoạn EI).

In [ ]:
counts = read_rows("itemset_counts.csv")
for encoding, title in (("nucleotide", "A/C/G/T"), ("purine_pyrimidine", "R/Y")):
    display(Markdown(f"**Mã hóa {title}**: số itemset (trong ngoặc: số itemset có từ 2 item trở lên)"))
    levels = [r["minsup"] for r in counts if r["encoding"] == encoding and r["class"] == "EI"]
    table = []
    for level in levels:
        row = [level]
        for cls in ("EI", "IE", "N"):
            r = next(r for r in counts
                     if r["encoding"] == encoding and r["class"] == cls and r["minsup"] == level)
            row.append(f'{r["itemsets"]} ({r["itemsets_2plus"]})')
        table.append(row)
    show_table(["minsup trong lớp", "EI (donor)", "IE (acceptor)", "N (đối chứng)"], table)

## Mẫu đặc trưng ở minsup chính

Bảng dưới chỉ giữ itemset tối đại: itemset phổ biến không nằm trong itemset phổ biến lớn hơn nào. Mọi itemset khác là tập con của chúng nên không thêm thông tin về mẫu. Mỗi mẫu được đếm lại ở cả ba lớp, kể cả lớp không trả về mẫu đó khi khai phá; ở chính lớp đã khai phá, số đếm lại phải bằng support Eclat trả về.

Cột motif viết mẫu từ vị trí -20 đến +10: dấu `|` là ranh giới, dấu `.` là vị trí không thuộc mẫu. Confidence là xác suất một đoạn chứa mẫu thuộc đúng lớp đó; nó phụ thuộc tỉ lệ lớp trong bộ dữ liệu nên chỉ dùng để so giữa các mẫu.

Mẫu chuẩn dùng ký hiệu IUPAC: `M` là A hoặc C, `R` là A hoặc G, `Y` là C hoặc T.

In [ ]:
patterns = read_rows("patterns.csv")
for encoding, title in (("nucleotide", "A/C/G/T, minsup 30%"), ("purine_pyrimidine", "R/Y, minsup 60%")):
    for cls in ("EI", "IE"):
        rows = [r for r in patterns
                if r["encoding"] == encoding and r["class"] == cls and r["maximal"] == "true"]
        rows.sort(key=lambda r: (-int(r["length"]), -int(r["support"])))
        display(Markdown(f"**{cls}, mã hóa {title}**: {len(rows)} itemset tối đại; "
                         "5 itemset dài nhất, cùng độ dài thì support cao hơn đứng trước"))
        show_table(
            ["itemset", "motif (-20..+10)", "tỉ lệ ở EI", "tỉ lệ ở IE", "tỉ lệ ở N", "confidence"],
            [[r["itemset"], f'`{r["motif"]}`', percent(r["freq_ei"]), percent(r["freq_ie"]),
              percent(r["freq_n"]), r["confidence"]] for r in rows[:5]],
        )

consensus = read_rows("consensus.csv")
display(Markdown("**Mẫu chuẩn trong `splice.theory`**: tỉ lệ đoạn khớp ở mỗi lớp"))
table = []
for pattern, start in dict.fromkeys((r["pattern"], r["start"]) for r in consensus):
    fraction = {r["class"]: r["fraction"] for r in consensus if r["pattern"] == pattern and r["start"] == start}
    table.append([f"`{pattern}`", start] + [percent(fraction[cls]) for cls in ("EI", "IE", "N")])
show_table(["mẫu", "bắt đầu ở vị trí", "EI", "IE", "N"], table)

## Phụ thuộc giữa hai vị trí

Hai item thường đi cùng nhau có thể chỉ vì cả hai đều phổ biến. Với mỗi 2-itemset phổ biến, chương trình lập bảng 2×2 (có cả hai, chỉ có item đầu, chỉ có item sau, không có item nào), tính lift và χ². Lift cho biết hai vị trí đi cùng (> 1) hay tránh nhau (< 1); χ² cho biết độ lệch so với giả định độc lập có vượt dao động ngẫu nhiên không.

Chỉ đọc χ² khi mọi ô kỳ vọng từ 5 trở lên. Ngưỡng được hiệu chỉnh Bonferroni theo số cặp *có thể* được kiểm: mọi cặp item đơn phổ biến ở hai vị trí khác nhau, đủ điều kiện kỳ vọng, dù đạt minsup hay không. Các cặp thực sự được tính đã được chọn ra theo chính số lần đồng xuất hiện, nên chia theo số cặp đó sẽ cho ngưỡng quá dễ.

In [ ]:
from statistics import NormalDist

family = read_rows("pair_family.csv")
candidates = sum(int(r["candidate_pairs"]) for r in family)
critical = NormalDist().inv_cdf(1 - 0.05 / candidates / 2) ** 2
pairs = read_rows("pair_dependence.csv")
eligible = [r for r in pairs if float(r["min_expected"]) >= 5]
significant = sorted((r for r in eligible if float(r["chi2"]) > critical), key=lambda r: -float(r["chi2"]))

print(f"Số cặp có thể kiểm: {candidates}; ngưỡng p < 0,05/{candidates}, tức χ² > {critical:.2f}")
print(f"Cặp có 2-itemset đạt minsup: {len(pairs)}; đủ điều kiện kỳ vọng ≥ 5: {len(eligible)}; qua ngưỡng: {len(significant)}")
show_table(
    ["mã hóa", "lớp", "cặp", "lift", "χ²"],
    [[r["encoding"], r["class"], f'{r["item_a"]}, {r["item_b"]}', r["lift"], r["chi2"]] for r in significant],
)

## Kiểm tái lập và hình

Ô dưới kiểm tra đủ 8 bảng CSV ở cả lần chạy mới và bản đã commit, rồi so SHA-256 từng bảng. Mã băm ghi trong `environment.json` cũng được đối chiếu với hai bảng dùng để tạo biểu đồ.

- `itemset_counts`: số itemset có từ 2 item trở lên theo minsup trong lớp. Đường N nằm ở 0 ở mọi mức.
- `position_enrichment`: mỗi cột là một item đơn đạt minsup 30%, chiều cao là tỉ lệ trong lớp chia tỉ lệ ở lớp N. Cột cao của donor dồn vào -3..+6 (mẫu `MAG|GTRAGT`); của acceptor là `CAG` ngay trước ranh giới và dãy C/T phía trước.

In [ ]:
import json

committed = repo / "results/applications/splice"
figures = repo / "results/figures/splice"


def lf_sha256(path):
    # Git trên Windows có thể đổi xuống dòng thành CRLF khi checkout; so trên bản LF.
    return hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()


names = (
    "itemset_counts.csv",
    "patterns.csv",
    "pair_dependence.csv",
    "pair_family.csv",
    "consensus.csv",
    "cleaning_sensitivity.csv",
    "raw_checks.csv",
    "non_canonical.csv",
)
missing = [
    str(folder / name)
    for folder in (committed, splice_out)
    for name in names
    if not (folder / name).is_file()
]
if missing:
    raise FileNotFoundError("Thiếu bảng CSV: " + ", ".join(missing))

different = [name for name in names if lf_sha256(committed / name) != lf_sha256(splice_out / name)]
if different:
    raise RuntimeError("Bảng khác bản đã commit: " + ", ".join(different))

provenance = json.loads((figures / "environment.json").read_text(encoding="utf-8"))["inputs_sha256"]
mismatched = [name for name in ("itemset_counts.csv", "patterns.csv")
              if provenance.get(name) != lf_sha256(splice_out / name)]
if mismatched:
    raise RuntimeError("SHA-256 đầu vào của biểu đồ không khớp: " + ", ".join(mismatched))

print(f"Cả {len(names)} bảng CSV khớp với bản đã commit.")
print("SHA-256 của các bảng dùng để vẽ cả hai biểu đồ đều khớp.")

display(Image(filename=str(figures / "itemset_counts.png")))
display(Image(filename=str(figures / "position_enrichment.png")))

## Diễn giải ứng dụng 2

- Eclat của nhóm tìm lại quy tắc GT-AG đã biết: GT ngay sau ranh giới có ở 98,8% đoạn donor nhưng chỉ 5,9% đoạn N; AG ngay trước ranh giới có ở 99,4% đoạn acceptor, 7,6% đoạn N.
- Trong dải minsup khảo sát, lớp đối chứng N không có itemset từ hai item trở lên đạt ngưỡng. Một số itemset đạt ngưỡng ở EI hoặc IE vẫn xuất hiện trong N với tần suất thấp hơn ngưỡng tương ứng.
- Chỉ có GT thì chưa đủ để nhận ra donor: đoạn có `+1G +2T` là donor với confidence 0,77; đoạn có cả cụm `-1G +1G +2T +4A +5G` thì 0,98. Mẫu này có ở 46,8% đoạn donor và không có ở đoạn N nào.
- Mẫu chuẩn đầy đủ `MAGGTRAGT` chỉ khớp 4,0% đoạn donor. Vị trí cắt nối thật chỉ khớp mẫu chuẩn từng phần; Eclat cho biết những phần nào thường đi cùng nhau và với tần suất bao nhiêu.
- Mã hóa R/Y làm lộ vùng giàu pyrimidin trước acceptor: `-7Y -6Y -5Y -3Y -2R -1R` có ở 64,6% đoạn acceptor, 1,4% đoạn N. Với mã hóa A/C/G/T, vùng này bị chia giữa C và T ở từng vị trí nên khó đạt minsup.
- Ở donor, một số vị trí phía exon và phía intron của mẫu chuẩn có xu hướng ít cùng khớp hơn mức độc lập (lift < 1, qua ngưỡng χ² đã hiệu chỉnh).

Giới hạn: đồng xuất hiện không chứng minh cơ chế sinh học; dữ liệu chỉ gồm linh trưởng từ GenBank 64.1, cửa sổ 60 nucleotide; confidence và lift phụ thuộc tỉ lệ lớp do người dựng bộ dữ liệu chọn, nên xếp hạng dùng tỉ lệ so với lớp N.

[Ghi chú thiết kế ứng dụng 2](../docs/splice_notes.md) trình bày từng lựa chọn, phương án khác và bằng chứng.